# SortVision: Waste Image Classification

Multi-class classification of waste photos into six categories: cardboard,
glass, metal, paper, plastic, and trash (used as an Other / General
bucket).

Dataset: [TrashBusters/combined](https://huggingface.co/datasets/TrashBusters/combined)
on Hugging Face, a harmonized merge of four public waste image datasets,
29,849 images with a pre-made train, validation, and test split.

This notebook covers data loading, exploratory data analysis, a
from-scratch CNN, training curves, full test-set evaluation (accuracy,
macro and weighted precision/recall/F1, confusion matrix, per-class
performance, most confused class pairs), and Grad-CAM explainability.

**Before running:** `pip install -r ../requirements.txt`. The dataset
downloads automatically from Hugging Face the first time it is loaded, no
manual download or folder setup needed.

## 1. Setup

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(".."))  # so `import config` and `from src...` work from notebooks/

import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf

import config
from src.data_loader import load_waste_dataset, get_class_names
from src.preprocessing import compute_class_weights
from src.dataset import make_dataset
from src.model import build_cnn_from_scratch, compile_model
from src.gradcam import compute_gradcam_heatmap, overlay_heatmap

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

np.random.seed(config.RANDOM_STATE)
tf.random.set_seed(config.RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Target dataset:", config.HF_DATASET_NAME)

## 2. Load the dataset

Downloads from Hugging Face on first run, then uses the local cache on
later runs. No manual folder setup required, unlike a Kaggle zip.

In [ ]:
ds = load_waste_dataset()

names = get_class_names(ds)
print("Splits:", {split: len(ds[split]) for split in ds})
print("Class names from dataset:", names)
print("Class names in config.py:", config.CLASS_NAMES)

if names != config.CLASS_NAMES:
    print("\nWARNING: dataset class order does not match config.CLASS_NAMES. "
          "Update config.py to match before trusting any results.")

## 3. Exploratory Data Analysis

### 3.1 Class distribution

In [ ]:
train_labels = ds["train"]["label"]
counts = collections.Counter(train_labels)
display_names = [config.DISPLAY_NAMES.get(n, n) for n in names]

plt.figure(figsize=(8, 4))
plt.bar(display_names, [counts.get(i, 0) for i in range(len(names))], color="#3F7D4F")
plt.title("Train Class Distribution")
plt.ylabel("Number of images")
plt.xticks(rotation=20, ha="right")
for i, name in enumerate(names):
    v = counts.get(i, 0)
    plt.text(i, v + 50, f"{v:,}", ha="center")
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, "class_distribution.png"), dpi=150)
plt.show()

print("Class balance (train):")
total = len(train_labels)
for i, name in enumerate(names):
    print(f"  {config.DISPLAY_NAMES.get(name, name):16s} {counts.get(i,0):6,}  ({100*counts.get(i,0)/total:5.1f}%)")

### 3.2 Sample images

In [ ]:
def show_sample_images(ds_split, class_names, display_names, n_per_class=3):
    fig, axes = plt.subplots(len(class_names), n_per_class, figsize=(2.2 * n_per_class, 2.2 * len(class_names)))
    labels_arr = np.array(ds_split["label"])
    for row, cls_idx in enumerate(range(len(class_names))):
        idxs = np.where(labels_arr == cls_idx)[0]
        chosen = np.random.choice(idxs, size=min(n_per_class, len(idxs)), replace=False)
        for col, idx in enumerate(chosen):
            img = ds_split[int(idx)]["image"]
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
        axes[row, 0].set_ylabel(display_names[row], fontsize=10)
        axes[row, 0].axis("on")
        axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    plt.suptitle("Sample Images per Class")
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUTS_DIR, "sample_images.png"), dpi=150)
    plt.show()

show_sample_images(ds["train"], names, display_names)

## 4. Build the tf.data pipelines

For a fast CPU run, subsample the training and validation sets. Set
`MAX_TRAIN_SAMPLES` and `MAX_VAL_SAMPLES` to `None` for the full dataset
on a GPU.

In [ ]:
MAX_TRAIN_SAMPLES = 5000
MAX_VAL_SAMPLES = 1000

train_split = ds["train"]
val_split = ds["validation"]
test_split = ds["test"]

if MAX_TRAIN_SAMPLES and len(train_split) > MAX_TRAIN_SAMPLES:
    train_split = train_split.shuffle(seed=config.RANDOM_STATE).select(range(MAX_TRAIN_SAMPLES))
if MAX_VAL_SAMPLES and len(val_split) > MAX_VAL_SAMPLES:
    val_split = val_split.shuffle(seed=config.RANDOM_STATE).select(range(MAX_VAL_SAMPLES))

print(f"Using for this run -> Train: {len(train_split):,}  Val: {len(val_split):,}  Test (untouched): {len(test_split):,}")

train_ds = make_dataset(train_split, config.IMAGE_SIZE, config.BATCH_SIZE, config.NUM_CLASSES, training=True)
val_ds = make_dataset(val_split, config.IMAGE_SIZE, config.BATCH_SIZE, config.NUM_CLASSES, training=False)
test_ds = make_dataset(test_split, config.IMAGE_SIZE, config.BATCH_SIZE, config.NUM_CLASSES, training=False)

## 5. Build the model

In [ ]:
model = build_cnn_from_scratch(config.NUM_CLASSES, config.IMAGE_SIZE, config.CHANNELS)
model = compile_model(model, config.LEARNING_RATE)
model.summary()

In [ ]:
class_weights = compute_class_weights(train_split["label"], config.NUM_CLASSES)
print("Class weights:", class_weights)

## 6. Train

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max",
        patience=config.EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", mode="max",
        factor=config.REDUCE_LR_FACTOR,
        patience=config.REDUCE_LR_PATIENCE,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
)

model.save(config.CNN_MODEL_PATH)
print(f"Saved trained model to {config.CNN_MODEL_PATH}")

## 7. Training curves: accuracy, loss, AUC

In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(hist_df["accuracy"], label="Train")
axes[0].plot(hist_df["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["loss"], label="Train")
axes[1].plot(hist_df["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(hist_df["auc"], label="Train")
axes[2].plot(hist_df["val_auc"], label="Validation")
axes[2].set_title("AUC")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.suptitle("Training Curves")
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, "training_curves.png"), dpi=150)
plt.show()

best_epoch = int(hist_df["val_accuracy"].idxmax()) + 1
print(f"Training ran {len(hist_df)} epochs, best val_accuracy at epoch {best_epoch}")
print(f"Best val_accuracy: {hist_df['val_accuracy'].max():.4f}")

## 8. Test set evaluation (untouched)

In [ ]:
y_true = np.array(test_split["label"])
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy: {acc:.4f}")
print(f"Macro    precision/recall/F1: {precision_macro:.4f} / {recall_macro:.4f} / {f1_macro:.4f}")
print(f"Weighted precision/recall/F1: {precision_weighted:.4f} / {recall_weighted:.4f} / {f1_weighted:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=display_names, zero_division=0))

### 8.1 Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 7))
disp = ConfusionMatrixDisplay(cm, display_labels=display_names)
disp.plot(cmap="Greens", ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

### 8.2 Per-class performance

In [ ]:
precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(y_true, y_pred, zero_division=0)

x = np.arange(len(display_names))
width = 0.25
plt.figure(figsize=(9, 5))
plt.bar(x - width, precision_per, width, label="Precision")
plt.bar(x, recall_per, width, label="Recall")
plt.bar(x + width, f1_per, width, label="F1")
plt.xticks(x, display_names, rotation=30, ha="right")
plt.ylim(0, 1)
plt.title("Per-Class Performance")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUTS_DIR, "per_class_performance.png"), dpi=150)
plt.show()

### 8.3 Most confused class pairs

In [ ]:
def most_confused_pairs(cm, class_names, top_n=5):
    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                pairs.append((class_names[i], class_names[j], int(cm[i, j])))
    pairs.sort(key=lambda p: p[2], reverse=True)
    return pairs[:top_n]

confused = most_confused_pairs(cm, display_names)
print("Most confused class pairs (true -> predicted, count):")
for true_cls, pred_cls, count in confused:
    print(f"  {true_cls} -> {pred_cls}: {count}")

## 9. Grad-CAM: what did the model look at?

Shows which part of an image most influenced the model's prediction, for
a few random test images.

In [ ]:
def show_gradcam_examples(model, test_split, class_names, display_names, n=4):
    labels_arr = np.array(test_split["label"])
    idxs = np.random.choice(len(test_split), size=n, replace=False)

    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 6.5))

    for col, idx in enumerate(idxs):
        example = test_split[int(idx)]
        original = example["image"].convert("RGB").resize((config.IMAGE_SIZE, config.IMAGE_SIZE))
        arr = np.expand_dims(np.asarray(original, dtype=np.float32), axis=0)

        true_label = display_names[example["label"]]

        heatmap, pred_idx = compute_gradcam_heatmap(model, arr)
        overlay = overlay_heatmap(original, heatmap)
        pred_label = display_names[pred_idx]

        axes[0, col].imshow(original)
        axes[0, col].set_title(f"True: {true_label}", fontsize=10)
        axes[0, col].axis("off")

        axes[1, col].imshow(overlay)
        axes[1, col].set_title(f"Pred: {pred_label}", fontsize=10)
        axes[1, col].axis("off")

    plt.suptitle("Grad-CAM: Original (top) vs Heatmap Overlay (bottom)")
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUTS_DIR, "gradcam_examples.png"), dpi=150)
    plt.show()

show_gradcam_examples(model, test_split, names, display_names)

## 10. Conclusion

Fill this in with real numbers after running the cells above. State
clearly whether this was a subsampled run (see Section 4's
`MAX_TRAIN_SAMPLES` and `MAX_VAL_SAMPLES`) or the full dataset, since the
two are not directly comparable and should never be presented as if they
were the same thing.

Note the dataset's two known gaps when writing up results: paper and
cardboard are separate classes here rather than merged, and there is no
dedicated Organic class, food and biological waste falls under trash
instead. Both are documented starting points, not errors.

Reasonable next steps: train on the full dataset with GPU acceleration,
run the transfer learning backbones (`--model_type transfer --backbone
mobilenet_v2`, etc.) and compare them against this from-scratch CNN,
merge in a dedicated Organic waste dataset, and consider quantizing the
deployed model to TensorFlow Lite for lighter hosting.

Disclaimer: this project is for learning and research purposes only. It
is not a guarantee of correct waste sorting.